# Nowcasting Net Asset Values: The Case of Private Equity

**A pedagogical implementation of Brown, Ghysels, Gredil (*The Review of Financial Studies*, 2023)**

---

## Table of Contents

1. [Introduction & Motivation](#1-introduction--motivation)
2. [The Data Generating Process](#2-the-data-generating-process)
3. [The Naive Nowcast](#3-the-naive-nowcast)
4. [State Space Model Formulation](#4-state-space-model-formulation)
5. [Kalman Filter & Smoother](#5-kalman-filter--smoother)
6. [Parameter Estimation](#6-parameter-estimation)
7. [Partial Imputation](#7-partial-imputation)
8. [Nowcasting Performance Metrics](#8-nowcasting-performance-metrics)
9. [Monte Carlo Experiments](#9-monte-carlo-experiments)
10. [Sensitivity Analysis](#10-sensitivity-analysis)
11. [Glossary & Appendix](#11-glossary--appendix)

---

## 1. Introduction & Motivation <a id="1-introduction--motivation"></a>

### The Private Equity Valuation Problem

Private equity (PE) funds report their Net Asset Values (NAVs) only **quarterly**, with a significant reporting lag. Between reports, the true value of the portfolio is unknown to investors (LPs). This creates several problems:

1. **Stale valuations**: NAVs reflect conditions from weeks or months ago, not today.
2. **Appraisal smoothing**: Fund managers deliberately or mechanically smooth reported NAVs, understating true volatility.
3. **Risk measurement**: Standard risk metrics (volatility, beta, correlation) are biased downward.
4. **Portfolio allocation**: LPs making allocation decisions operate with outdated information.

### The Nowcasting Solution

Brown, Ghysels, and Gredil propose a **state space model (SSM)** that combines:
- **Quarterly NAV reports** (sparse, noisy, smoothed)
- **Irregular fund distributions** (cash flows to LPs)
- **Weekly comparable public asset returns** (always observed)

Using a **Kalman filter**, the model extracts the latent true fund return series at **weekly** frequency. The key insight: even though NAVs are only observed quarterly, the comparable asset returns provide weekly information about the fund's likely performance.

### What This Notebook Covers

We implement the full methodology using **simulated data** with known parameters, which allows us to:
- Validate the SSM against ground truth
- Understand each component of the model
- See how the Kalman filter extracts signal from sparse observations

In [ ]:
# Standard imports
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

# Our modules
from src.simulation import FundParams, SimulatedFund, simulate_fund
from src.naive import compute_naive_nav, compute_naive_returns
from src.ssm import build_ssm_matrices, build_observation_vector
from src.kalman import kalman_filter, kalman_smoother, extract_returns
from src.estimation import estimate_fund_params, profile_likelihood_grid, EstimationResult
from src.metrics import (
    compute_pme_series, compute_insample_rmse, compute_oos_rmse,
    compute_improvement_rate, return_autocorrelation,
)
from src.visualization import (
    plot_fund_lifecycle, plot_naive_vs_true, plot_ssm_nowcast,
    plot_return_comparison, plot_profile_likelihood, plot_pme_series,
    plot_monte_carlo_results, plot_sensitivity,
)

# Plotting defaults
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print('All modules loaded successfully.')

---

## 2. The Data Generating Process <a id="2-the-data-generating-process"></a>

We simulate a PE fund using the paper's DGP (equations 1-9). The simulation creates both **latent** (unobservable) and **observed** time series.

### Fund Return Process (eq 1)

$$R_t = (\alpha + \beta(R_{m,t} - 1) + 1) \cdot e^{\eta_t}, \quad \eta_t \sim N(0, F^2 h_t)$$

where:
- $\alpha$ = weekly excess return (alpha)
- $\beta$ = market risk exposure
- $R_{m,t}$ = weekly gross market return
- $F$ = idiosyncratic volatility scale factor
- $h_t$ = GARCH(1,1) conditional variance (time-varying volatility proxy)

### Comparable Asset Returns (eq 2)

$$R_{c,t} = \exp(r_t \cdot \beta_c + \psi + \eta_{c,t}), \quad \eta_{c,t} \sim N(0, F_c^2 h_t)$$

The comparable asset (e.g., a sector ETF) is correlated with the fund's true returns, providing weekly information.

### True Asset Values (eq 3)

$$V_t = V_{t-1} \cdot R_t - D_t + C_t$$

### NAV Smoothing (eqs 5-6, 8)

$$\bar{r}_{0:t} = (1 - \lambda(\cdot)_t) \cdot r_{0:t} + \lambda(\cdot)_t \cdot \bar{r}_{0:t-1}$$

$$NAV_t = \exp(\bar{r}_{0:t} - m_t + n_t), \quad n_t \sim N(0, \sigma_n^2)$$

where $\lambda(\cdot)_t = \lambda \cdot (1 - w_t)$ controls how much the reported NAV lags behind the true value.

In [ ]:
# Define fund parameters (matching the paper's Figure 1 simulation)
params = FundParams(
    alpha=0.0,           # No excess return (for clean simulation)
    beta=1.19,           # Market exposure (buyout-like)
    F=2.0,               # Idiosyncratic vol scale
    beta_c=0.85,         # Comparable asset loading
    psi=0.001,           # Comparable asset drift
    Fc=1.0,              # Comparable idiosyncratic vol scale
    lam=0.90,            # Smoothing parameter (strong smoothing)
    sigma_n=0.05,        # NAV noise std dev
    delta=0.03,          # Distribution density parameter
    sigma_d=0.10,        # Distribution noise std dev
    T_weeks=562,         # ~10.8 years
    fund_size=100.0,     # $100M commitment
    seed=42,
)

# Simulate the fund
fund = simulate_fund(params)

print(f'Fund life: {fund.T} weeks ({fund.T/52:.1f} years)')
print(f'NAV reports: {fund.is_nav_week.sum()} quarters')
print(f'Distributions: {fund.is_dist_week.sum()} events')
print(f'Capital calls: {(fund.C > 0).sum()} events')
print(f'\nTrue parameters:')
print(f'  α = {params.alpha:.4f}, β = {params.beta:.2f}')
print(f'  F = {params.F:.1f}, λ = {params.lam:.2f}')
print(f'  σn = {params.sigma_n:.3f}, σd = {params.sigma_d:.3f}')

In [ ]:
# Visualize the simulated fund lifecycle
fig, axes = plot_fund_lifecycle(fund)
plt.show()

### Key Observations

- **Top-left**: The true fund value (blue) fluctuates significantly, but reported NAVs (red dots) appear smoother and lagged. This is the **appraisal smoothing** problem.
- **Top-right**: Smoothed cumulative returns (red) visibly lag behind true cumulative returns (blue), confirming the smoothing parameter $\lambda = 0.90$.
- **Bottom-left**: Capital calls are concentrated in the first ~3 years; distributions start around year 3 and increase over time.
- **Bottom-right**: Weekly returns show significant idiosyncratic volatility, with the $F \sqrt{h_t}$ bands capturing the time-varying conditional volatility.

---

## 3. The Naive Nowcast <a id="3-the-naive-nowcast"></a>

The simplest approach to filling in weekly NAVs is **Rc-interpolation** (equation A.10).

### The Idea

Between two quarterly NAV reports at times $q^-$ and $q$:

1. Start from the last reported NAV: $NAV_{q^-}$
2. Grow it each week by the comparable asset return: $\tilde{V}_t = \tilde{V}_{t-1} \cdot R_{c,t}$
3. Adjust for interim cash flows (distributions and calls)
4. Apply a quarter-specific drift $\psi_q$ that reconciles with the next NAV report

### Limitations

- Only uses **one** source of information (comparable returns)
- Ignores the smoothing structure of NAV reporting
- No probabilistic framework (no uncertainty estimates)
- Cannot incorporate distribution information

In [ ]:
# Compute the naive Rc-interpolated NAV
nav_naive = compute_naive_nav(
    NAV_reported=fund.NAV_reported,
    rc=fund.rc,
    C=fund.C,
    D=fund.D,
    is_nav_week=fund.is_nav_week,
)

# Compute implied returns
returns_naive = compute_naive_returns(nav_naive, fund.C, fund.D)

# Plot comparison
fig, axes = plot_naive_vs_true(fund, nav_naive)
plt.show()

# Summary statistics
valid = ~np.isnan(nav_naive)
tracking_error = (nav_naive[valid] - fund.V_true[valid]) / fund.V_true[valid]
print(f'\nNaive nowcast tracking error:')
print(f'  Mean: {tracking_error.mean():.2%}')
print(f'  Std:  {tracking_error.std():.2%}')
print(f'  RMSE: {np.sqrt(np.mean(tracking_error**2)):.2%}')

The naive nowcast tracks the general trend but misses short-term fluctuations and has significant tracking error. The SSM approach will improve on this by combining **all** available information sources.

---

## 4. State Space Model Formulation <a id="4-state-space-model-formulation"></a>

The SSM casts the PE fund nowcasting problem into the standard linear Gaussian form:

### Observation Equation (eq A.4)

$$y_t - \Gamma_t x_t = Z_t s_t + \varepsilon_t, \quad \varepsilon_t \sim N(0, H_t)$$

### Transition Equation (eq A.5)

$$s_{t+1} = G_t s_t + V_t \eta_t, \quad \eta_t \sim N(0, Q_t)$$

### State Vector

$$s_t = \begin{bmatrix} r_t \\ r_{0:t-1} \\ \bar{r}_{0:t-2} \\ 1 \end{bmatrix}$$

where:
- $r_t$ = current week's true log fund return (latent)
- $r_{0:t-1}$ = cumulative log return through last week
- $\bar{r}_{0:t-2}$ = smoothed cumulative log return (lagged)
- $1$ = constant for intercept terms

### Observation Vector

$$y_t = \begin{bmatrix} d_{t-1} \\ nav_{t-2} \\ r_{c,t} \end{bmatrix}$$

**Crucially, $d_{t-1}$ and $nav_{t-2}$ are often missing!** The Kalman filter handles this by dropping the corresponding rows from $Z_t$ and $H_t$.

- $d_{t-1}$: only present when a distribution occurred at $t-1$ (~25 of 562 weeks)
- $nav_{t-2}$: only present every 13th week (quarterly reporting, lagged 2 weeks)
- $r_{c,t}$: **always** observed (comparable returns are weekly)

In [ ]:
# Demonstrate SSM matrix construction for different scenarios

# Week with only comparable return (most common case)
t_normal = 50
ssm_normal = build_ssm_matrices(
    t=t_normal, alpha=params.alpha, beta=params.beta,
    beta_c=params.beta_c, psi=params.psi,
    F=params.F, Fc=params.Fc, lam=params.lam, delta=params.delta,
    sigma_n=params.sigma_n, sigma_d=params.sigma_d,
    Rm_t=fund.rm[t_normal], h_t=fund.h_t[t_normal],
    lambda_t=fund.lambda_t[t_normal], delta_t=fund.delta_t[t_normal],
    m_hat=fund.m_t, has_distribution=False, has_nav=False,
)

print('=== Normal week (only comparable return observed) ===')
print(f'Observations: {ssm_normal.n_obs}')
print(f'Z shape: {ssm_normal.Z.shape}')
print(f'Z =\n{ssm_normal.Z}')
print(f'\nG (transition) =\n{ssm_normal.G}')
print(f'\nH (obs noise cov) =\n{ssm_normal.H}')

# Find a NAV week
nav_week_idx = np.where(fund.is_nav_week)[0][3]  # 4th NAV report
ssm_nav = build_ssm_matrices(
    t=nav_week_idx + 2, alpha=params.alpha, beta=params.beta,
    beta_c=params.beta_c, psi=params.psi,
    F=params.F, Fc=params.Fc, lam=params.lam, delta=params.delta,
    sigma_n=params.sigma_n, sigma_d=params.sigma_d,
    Rm_t=fund.rm[nav_week_idx + 2], h_t=fund.h_t[nav_week_idx + 2],
    lambda_t=fund.lambda_t[nav_week_idx + 2], delta_t=fund.delta_t[nav_week_idx + 2],
    m_hat=fund.m_t, has_distribution=False, has_nav=True,
)

print(f'\n=== NAV week (comparable + NAV observed) ===')
print(f'Observations: {ssm_nav.n_obs}')
print(f'Z shape: {ssm_nav.Z.shape}')
print(f'Z =\n{ssm_nav.Z}')
print(f'H =\n{ssm_nav.H}')

### Key Points About the SSM

1. **On most weeks** (no distribution, no NAV), the Kalman filter only uses comparable returns ($r_{c,t}$) to update the state estimate. The Z matrix has just 1 row.

2. **On NAV weeks**, the filter gets extra information from the reported NAV. The Z matrix has 2 rows, and the NAV observation helps correct the cumulative return estimate.

3. **On distribution weeks**, the distribution amount provides yet another signal. All three observations may be present.

4. The **transition matrix G** encodes the return dynamics: how current returns cumulate, how smoothing works, and how the market factor drives expected returns.

5. The **observation noise H** is diagonal: the three types of noise (distribution, NAV, comparable) are independent.

---

## 5. Kalman Filter & Smoother <a id="5-kalman-filter--smoother"></a>

### Forward Recursion: Kalman Filter (eq A.2)

The Kalman filter processes observations forward in time. At each step:

1. **Predict**: Project the state forward using the transition equation
   $$s_{t|t-1} = G_t s_{t-1|t-1}$$

2. **Update**: Incorporate the new observation to refine the estimate
   $$s_{t|t} = s_{t|t-1} + K_t (y_t - Z_t s_{t|t-1})$$

where $K_t$ is the **Kalman gain** — the optimal weight on new information vs. prior estimate.

### Backward Recursion: Kalman Smoother (eq A.3)

The smoother uses **future** observations to improve past estimates:

$$s_{t|T}^{smooth} = s_{t|t}^{filter} + P_{t|t} \cdot b_{t-1}$$

This is especially valuable because NAV reports at quarter-end contain information about what the fund was worth during the preceding quarter.

Let's run the Kalman filter and smoother using the **true** parameters (to validate the methodology before estimation).

In [ ]:
# Run Kalman filter with TRUE parameters
kf_result = kalman_filter(
    T=fund.T,
    alpha=params.alpha, beta=params.beta,
    beta_c=params.beta_c, psi=params.psi,
    F=params.F, Fc=params.Fc, lam=params.lam, delta=params.delta,
    sigma_n=params.sigma_n, sigma_d=params.sigma_d,
    Rm=fund.rm, rc_log=fund.rc_log, h_t=fund.h_t,
    D=fund.D, C=fund.C, NAV_reported=fund.NAV_reported,
    is_dist_week=fund.is_dist_week, is_nav_week=fund.is_nav_week,
    lambda_t=fund.lambda_t, delta_t=fund.delta_t,
    m_hat=fund.m_t,
)

print(f'Log-likelihood: {kf_result.log_likelihood:.1f}')

# Run smoother
kf_result = kalman_smoother(
    kf_result,
    alpha=params.alpha, beta=params.beta,
    beta_c=params.beta_c, psi=params.psi,
    F=params.F, Fc=params.Fc, lam=params.lam, delta=params.delta,
    sigma_n=params.sigma_n, sigma_d=params.sigma_d,
    Rm=fund.rm, h_t=fund.h_t,
    D=fund.D, NAV_reported=fund.NAV_reported,
    is_dist_week=fund.is_dist_week, is_nav_week=fund.is_nav_week,
    lambda_t=fund.lambda_t, delta_t=fund.delta_t,
    m_hat=fund.m_t,
)

# Extract returns
r_filtered, r_cum_filtered = extract_returns(kf_result, use_smoothed=False)
r_smoothed, r_cum_smoothed_est = extract_returns(kf_result, use_smoothed=True)

# Compare to truth
rmse_filtered = np.sqrt(np.mean((r_filtered - fund.r_true_log)**2))
rmse_smoothed = np.sqrt(np.mean((r_smoothed - fund.r_true_log)**2))
corr_filtered = np.corrcoef(r_filtered, fund.r_true_log)[0, 1]
corr_smoothed = np.corrcoef(r_smoothed, fund.r_true_log)[0, 1]

print(f'\nWeekly return recovery (with true params):')
print(f'  Filtered — RMSE: {rmse_filtered:.4f}, Correlation: {corr_filtered:.4f}')
print(f'  Smoothed — RMSE: {rmse_smoothed:.4f}, Correlation: {corr_smoothed:.4f}')

In [ ]:
# Visualize SSM nowcast results
fig, axes = plot_ssm_nowcast(fund, kf_result, nav_naive)
plt.show()

In [ ]:
# Compare return series
fig, axes = plot_return_comparison(
    true_returns=fund.r_true_log,
    filtered_returns=r_filtered,
    naive_returns=np.log(returns_naive) if returns_naive is not None else None,
)
plt.show()

### Takeaways

- The Kalman filter successfully extracts the latent true returns from sparse observations.
- **Smoothed** estimates (using future data) are better than **filtered** (causal) estimates.
- The autocorrelation of filtered returns should be much lower than that of smoothed NAV returns, indicating successful "unsmoothing."
- On NAV reporting weeks, the filter incorporates a large information update (visible as jumps in the Kalman gain).

---

## 6. Parameter Estimation <a id="6-parameter-estimation"></a>

In practice, we don't know the true parameters. The paper uses a **two-step estimation procedure**:

### Step 1: Profile Likelihood Grid (Sections 2.4.1-2.4.2)

Search over a 15x15 grid of $(\alpha, \beta)$ values. For each grid point:
1. Fix $(\alpha, \beta)$ to the grid values
2. Estimate the remaining 8 parameters via MLE: $(\delta, \lambda, F, F_c, \sigma_n, \sigma_d, \beta_c, \psi)$
3. Compute the **penalized log-likelihood**

Select the $(\alpha, \beta)$ with the highest penalized log-likelihood.

### Step 2: EM-like Iterations

Fix $(\alpha, \beta)$ at the optimum and iteratively:
1. **E-step**: Run Kalman filter to get state estimates
2. **Update**: Recompute the mapping function $\hat{m}_t$ from filtered states
3. **M-step**: Re-estimate remaining parameters
4. Repeat until convergence

### Running the Estimation

We use a coarser grid (7x7) for speed in this demonstration.

In [ ]:
# Run the full estimation procedure
# Using a coarser grid for speed (7x7 instead of 15x15)
est_result = estimate_fund_params(
    fund=fund,
    n_alpha=7,
    n_beta=7,
    n_em_iterations=3,
    penalty_weight=0.5,
    verbose=True,
)

print(f'\n=== Estimation Results ===')
print(f'True   α = {params.alpha:.4f},  Estimated α = {est_result.alpha:.4f}')
print(f'True   β = {params.beta:.3f},  Estimated β = {est_result.beta:.3f}')
print(f'True   F = {params.F:.2f},   Estimated F = {est_result.F:.2f}')
print(f'True   λ = {params.lam:.3f},  Estimated λ = {est_result.lam:.3f}')
print(f'True  σn = {params.sigma_n:.4f}, Estimated σn = {est_result.sigma_n:.4f}')
print(f'True  σd = {params.sigma_d:.4f}, Estimated σd = {est_result.sigma_d:.4f}')
print(f'\nConverged: {est_result.converged}, Iterations: {est_result.n_iterations}')
print(f'Penalized LL: {est_result.penalized_ll:.1f}')

In [ ]:
# Plot profile likelihood surface
if est_result.ll_grid is not None:
    fig, ax = plot_profile_likelihood(
        alpha_grid=est_result.alpha_grid,
        beta_grid=est_result.beta_grid,
        ll_grid=est_result.ll_grid,
        true_alpha=params.alpha,
        true_beta=params.beta,
    )
    plt.show()

### Interpretation

- The profile likelihood surface should have a clear peak near the true parameter values.
- The $(\alpha, \beta)$ identification comes from combining multiple data sources: comparable returns pin down $\beta$, while the cash flow timing helps identify $\alpha$.
- The penalized likelihood prevents extreme parameter values that would produce unrealistic return paths.

---

## 7. Partial Imputation <a id="7-partial-imputation"></a>

When fund-level data is too sparse (e.g., very few distributions, short track record), the full profile grid may not identify parameters well. The paper proposes **partial imputation**:

1. **β-anchoring**: Set $\beta$ to the peer-group median (e.g., median of all buyout funds of the same vintage).
2. **Reduced estimation**: Only profile over $\alpha$ (1D grid), with $\beta$ fixed.
3. **Imputed starting values**: Use peer-fund estimates of $F$, $\lambda$ as initial guesses.

This dramatically reduces the computational burden and improves estimation stability.

In [ ]:
from src.estimation import partial_imputation

# Simulate a "peer estimate" of beta (with some noise)
peer_beta = params.beta + 0.05  # Slightly off from true value

# Run partial imputation estimation
imputed_result = partial_imputation(
    fund=fund,
    peer_beta=peer_beta,
    peer_F=2.0,
    peer_lam=0.9,
    n_alpha=15,
    penalty_weight=0.5,
    verbose=False,
)

if imputed_result is not None:
    print('=== Partial Imputation Results ===')
    print(f'Anchored β = {imputed_result.beta:.3f} (peer estimate; true = {params.beta:.3f})')
    print(f'Estimated α = {imputed_result.alpha:.4f} (true = {params.alpha:.4f})')
    print(f'Estimated F = {imputed_result.F:.2f} (true = {params.F:.2f})')
    print(f'Estimated λ = {imputed_result.lam:.3f} (true = {params.lam:.3f})')
    print(f'Penalized LL: {imputed_result.penalized_ll:.1f}')
else:
    print('Partial imputation failed.')

---

## 8. Nowcasting Performance Metrics <a id="8-nowcasting-performance-metrics"></a>

The paper evaluates nowcast quality using **Public Market Equivalent (PME)**:

$$PME(\theta, T)_{0:t} = \frac{\sum_{s=0}^t D_s \cdot \text{df}(s,t) + V_t}{\sum_{s=0}^t C_s \cdot \text{df}(s,t)}$$

where $\text{df}(s,t) = \exp(r_{0:t} - r_{0:s})$ is the discount factor using model-implied returns.

If the model returns are correct, $PME = 1$. Deviations measure pricing error.

### Three RMSE Metrics

| Metric | Window | Description |
|--------|--------|-------------|
| **In-Sample** | $[0, T]$ | Error within the estimation period |
| **OOS** | $[\tau, T]$ | Error using only post-estimation data |
| **Hybrid** | $[0, T]$ with OOS NAVs | Uses full cash flow history but OOS evaluation |

In [ ]:
# Compute PME series for SSM and naive nowcasts

# SSM cumulative returns (from estimated parameters)
if est_result is not None and est_result.kalman_result is not None:
    _, r_cum_est = extract_returns(est_result.kalman_result, use_smoothed=False)
else:
    r_cum_est = r_cum_filtered

# PME with SSM returns
pme_ssm = compute_pme_series(r_cum_est, fund.m_t, fund.C, fund.D)

# PME with true returns (gold standard)
pme_true = compute_pme_series(fund.r_cum_log, fund.m_t, fund.C, fund.D)

# Plot
fig, ax = plot_pme_series(pme_ssm, tau_cutoff=fund.T // 2)
# Add true PME
years = np.arange(fund.T) / 52
ax.plot(years, pme_true, 'b--', alpha=0.5, label='True returns PME')
ax.legend()
plt.show()

# RMSE metrics
tau_half = fund.T // 2
rmse_is = compute_insample_rmse(r_cum_est, fund.m_t, fund.C, fund.D, 52, tau_half)
rmse_oos = compute_oos_rmse(r_cum_est, fund.m_t, fund.C, fund.D, tau_half)

print(f'\n=== RMSE Metrics ===')
print(f'In-Sample RMSE (weeks 52-{tau_half}): {rmse_is:.4f}')
print(f'OOS RMSE (weeks {tau_half}-{fund.T}):     {rmse_oos:.4f}')

---

## 9. Monte Carlo Experiments <a id="9-monte-carlo-experiments"></a>

To assess the reliability of the estimation procedure, we simulate a panel of funds with the same true parameters but different random seeds, estimate parameters for each, and examine:

1. **Parameter recovery**: Do estimates center on the true values?
2. **Nowcast RMSE**: How often does the SSM beat the naive nowcast?

We use a small panel (10 funds) with a coarse grid for computational feasibility.

In [ ]:
# Monte Carlo with a small panel
n_funds = 10
beta_estimates = []
lam_estimates = []
F_estimates = []
ssm_rmses = []
naive_rmses = []

print('Running Monte Carlo experiments...')
for i in range(n_funds):
    # Simulate fund with different seed
    mc_params = FundParams(
        alpha=0.0, beta=1.19, F=2.0, beta_c=0.85, psi=0.001, Fc=1.0,
        lam=0.90, sigma_n=0.05, delta=0.03, sigma_d=0.10,
        T_weeks=300,  # Shorter for speed
        fund_size=100.0, seed=100 + i,
    )
    mc_fund = simulate_fund(mc_params)
    
    try:
        # Estimate with partial imputation (faster)
        mc_result = partial_imputation(
            fund=mc_fund, peer_beta=1.19,
            n_alpha=7, penalty_weight=0.5,
        )
        
        if mc_result is not None:
            beta_estimates.append(mc_result.beta)
            lam_estimates.append(mc_result.lam)
            F_estimates.append(mc_result.F)
            
            # Compute RMSE
            _, r_cum_mc = extract_returns(mc_result.kalman_result, use_smoothed=False)
            rmse_mc = compute_insample_rmse(r_cum_mc, mc_fund.m_t, mc_fund.C, mc_fund.D, 52, mc_fund.T - 1)
            ssm_rmses.append(rmse_mc)
            
            # Naive baseline
            nav_naive_mc = compute_naive_nav(
                mc_fund.NAV_reported, mc_fund.rc, mc_fund.C, mc_fund.D, mc_fund.is_nav_week
            )
            valid = ~np.isnan(nav_naive_mc)
            if valid.sum() > 0:
                naive_te = np.sqrt(np.mean(((nav_naive_mc[valid] - mc_fund.V_true[valid]) / mc_fund.V_true[valid])**2))
            else:
                naive_te = np.nan
            naive_rmses.append(naive_te)
            
            print(f'  Fund {i+1}: α̂={mc_result.alpha:.4f}, β̂={mc_result.beta:.2f}, '
                  f'λ̂={mc_result.lam:.3f}, F̂={mc_result.F:.2f}')
        else:
            print(f'  Fund {i+1}: estimation failed')
    except Exception as e:
        print(f'  Fund {i+1}: error - {e}')

# Summary
if len(beta_estimates) > 0:
    print(f'\n=== Monte Carlo Summary ({len(beta_estimates)} successful) ===')
    print(f'β: mean={np.mean(beta_estimates):.3f}, std={np.std(beta_estimates):.3f} (true=1.19)')
    print(f'λ: mean={np.mean(lam_estimates):.3f}, std={np.std(lam_estimates):.3f} (true=0.90)')
    print(f'F: mean={np.mean(F_estimates):.2f}, std={np.std(F_estimates):.2f} (true=2.00)')

In [ ]:
# Plot Monte Carlo results
if len(beta_estimates) > 2:
    fig, axes = plot_monte_carlo_results(
        param_estimates={'beta': np.array(beta_estimates), 'lam': np.array(lam_estimates)},
        true_params={'beta': 1.19, 'lam': 0.90},
        ssm_rmse_list=np.array(ssm_rmses),
        naive_rmse_list=np.array(naive_rmses),
    )
    plt.show()

---

## 10. Sensitivity Analysis <a id="10-sensitivity-analysis"></a>

How sensitive are the nowcasts to key parameters? We fix all parameters at their true values except one and vary it across a range.

In [ ]:
# Sensitivity to beta
beta_range = np.linspace(0.5, 2.0, 11)
rmse_by_beta = []

for beta_val in beta_range:
    kf_sens = kalman_filter(
        T=fund.T,
        alpha=params.alpha, beta=beta_val,
        beta_c=params.beta_c, psi=params.psi,
        F=params.F, Fc=params.Fc, lam=params.lam, delta=params.delta,
        sigma_n=params.sigma_n, sigma_d=params.sigma_d,
        Rm=fund.rm, rc_log=fund.rc_log, h_t=fund.h_t,
        D=fund.D, C=fund.C, NAV_reported=fund.NAV_reported,
        is_dist_week=fund.is_dist_week, is_nav_week=fund.is_nav_week,
        lambda_t=fund.lambda_t, delta_t=fund.delta_t,
        m_hat=fund.m_t,
    )
    r_f, r_cum_f = extract_returns(kf_sens, use_smoothed=False)
    rmse = np.sqrt(np.mean((r_f - fund.r_true_log)**2))
    rmse_by_beta.append(rmse)

fig, ax = plot_sensitivity('β', beta_range, np.array(rmse_by_beta), baseline_value=params.beta)
plt.show()

# Sensitivity to lambda
lam_range = np.linspace(0.1, 0.99, 11)
rmse_by_lam = []

for lam_val in lam_range:
    kf_sens = kalman_filter(
        T=fund.T,
        alpha=params.alpha, beta=params.beta,
        beta_c=params.beta_c, psi=params.psi,
        F=params.F, Fc=params.Fc, lam=lam_val, delta=params.delta,
        sigma_n=params.sigma_n, sigma_d=params.sigma_d,
        Rm=fund.rm, rc_log=fund.rc_log, h_t=fund.h_t,
        D=fund.D, C=fund.C, NAV_reported=fund.NAV_reported,
        is_dist_week=fund.is_dist_week, is_nav_week=fund.is_nav_week,
        lambda_t=fund.lambda_t, delta_t=fund.delta_t,
        m_hat=fund.m_t,
    )
    r_f, r_cum_f = extract_returns(kf_sens, use_smoothed=False)
    rmse = np.sqrt(np.mean((r_f - fund.r_true_log)**2))
    rmse_by_lam.append(rmse)

fig, ax = plot_sensitivity('λ', lam_range, np.array(rmse_by_lam), baseline_value=params.lam)
plt.show()

### Key Findings

- The nowcast RMSE is minimized near the **true** parameter values, confirming the model's identification.
- **β sensitivity**: Misspecifying β (market exposure) has a large impact because it affects the systematic component of every weekly return.
- **λ sensitivity**: The smoothing parameter affects how much weight the filter puts on NAV observations vs. comparable returns.

---

## 11. Glossary & Appendix <a id="11-glossary--appendix"></a>

### Key Concepts

| Concept | Brief Description | Link |
|---------|-------------------|------|
| **State Space Model (SSM)** | A framework for modeling time series with latent (hidden) states. Combines an observation equation (relating data to states) with a transition equation (evolving states over time). | [Wikipedia](https://en.wikipedia.org/wiki/State-space_representation) |
| **Kalman Filter** | Optimal recursive algorithm for estimating hidden states in a linear Gaussian SSM. Forward pass that processes data sequentially. | [Wikipedia](https://en.wikipedia.org/wiki/Kalman_filter) |
| **Kalman Smoother** | Backward pass that refines filtered estimates using future observations. Produces minimum-variance state estimates using all available data. | [Wikipedia](https://en.wikipedia.org/wiki/Kalman_filter#Rauch%E2%80%93Tung%E2%80%93Striebel_smoother) |
| **GARCH(1,1)** | Generalized Autoregressive Conditional Heteroskedasticity. Models time-varying volatility where current variance depends on past squared returns and past variance. | [Wikipedia](https://en.wikipedia.org/wiki/Autoregressive_conditional_heteroskedasticity#GARCH) |
| **Maximum Likelihood (MLE)** | Estimation method that finds parameters maximizing the probability of observing the data. The Kalman filter's innovation decomposition provides the likelihood. | [Wikipedia](https://en.wikipedia.org/wiki/Maximum_likelihood_estimation) |
| **Profile Likelihood** | Technique for estimating parameters in stages: fix some parameters, optimize over the rest. Reduces the dimension of the search space. | [Wikipedia](https://en.wikipedia.org/wiki/Likelihood_function#Profile_likelihood) |
| **EM Algorithm** | Expectation-Maximization: iteratively estimate latent variables (E-step) and parameters (M-step). Used here for the mapping function $m_t$. | [Wikipedia](https://en.wikipedia.org/wiki/Expectation%E2%80%93maximization_algorithm) |
| **NAV Smoothing / Appraisal Bias** | Systematic understatement of PE fund return volatility due to stale or smoothed valuations. The $\lambda$ parameter controls the degree of smoothing. | [Wikipedia](https://en.wikipedia.org/wiki/Appraisal_bias) |
| **Public Market Equivalent (PME)** | Performance metric comparing PE fund cash flows to what a public market investment would have returned. PME = 1 means the fund matched public markets. | [Wikipedia](https://en.wikipedia.org/wiki/Public_market_equivalent) |
| **Nowcasting** | Predicting the current state of a variable before official data is released. Borrowed from macroeconomics (GDP nowcasting). | [Wikipedia](https://en.wikipedia.org/wiki/Nowcasting_(economics)) |
| **MIDAS** | Mixed Data Sampling — econometric technique for combining data at different frequencies (e.g., weekly market returns + quarterly NAVs). | [Wikipedia](https://en.wikipedia.org/wiki/Mixed-data_sampling) |
| **Penalized Likelihood** | MLE with a penalty term discouraging extreme parameter values. Related to ridge regression. Improves estimation stability with limited data. | [Wikipedia](https://en.wikipedia.org/wiki/Penalized_likelihood) |

### Paper Reference

Brown, G. W., Ghysels, E., & Gredil, O. R. (2023). Nowcasting Net Asset Values: The Case of Private Equity. *The Review of Financial Studies*, 36(3), 945-986.

### Key Equations Summary

| Equation | Description | Number |
|----------|-------------|--------|
| $R_t = (\alpha + \beta(R_{m,t}-1)+1)e^{\eta_t}$ | Fund return process | (1) |
| $R_{c,t} = \exp(r_t\beta_c + \psi + \eta_{c,t})$ | Comparable asset returns | (2) |
| $V_t = V_{t-1}R_t - D_t + C_t$ | Value evolution | (3) |
| $R_{0:t} = \prod_{\tau=1}^t R_\tau$ | Cumulative returns | (4) |
| $\bar{r}_{0:t} = (1-\lambda(\cdot)_t)r_{0:t} + \lambda(\cdot)_t\bar{r}_{0:t-1}$ | NAV smoothing | (5) |
| $NAV_t = \exp(\bar{r}_{0:t} - m_t + n_t)$ | Reported NAV | (6) |
| $D_t = \delta(\cdot)_t(V_t+D_t)e^{d_t}$ | Distribution rule | (7) |
| $\lambda(\cdot)_t = \lambda(1-w_t)$ | Smoothing function | (8) |
| $\delta(\cdot)_t = \min(0.99, \delta \cdot t_y)$ | Distribution density | (9) |

---

*This notebook was created as a pedagogical tool. For production use, refer to the original paper and consider robustness checks beyond this prototype.*